# ✂️ ClipForge — free in your browser

Paste a YouTube link, get the most intense / funny / insightful moments as **vertical clips with animated captions**. Runs on Google's free GPU — nothing to install.

**How to use:** Runtime ▸ *Change runtime type* ▸ **T4 GPU** (optional but ~10× faster), then run the cells top to bottom (▶ on each, or Runtime ▸ Run all).

Source code & local install: https://github.com/WHERESHOULDISTART/clipforge

In [ ]:
#@title 1️⃣ Install ClipForge (≈1–2 min)
import os, subprocess, sys

def _gpu_name():
    try:
        return subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True, timeout=20).stdout.strip()
    except (FileNotFoundError, subprocess.SubprocessError, OSError):
        return ''  # CPU-only runtime: nvidia-smi doesn't exist
gpu = _gpu_name()
HAS_GPU = bool(gpu)

if not os.path.exists('/content/clipforge'):
    !git clone -q https://github.com/WHERESHOULDISTART/clipforge /content/clipforge
%cd -q /content/clipforge
!git pull -q
!apt-get -qq install -y ffmpeg > /dev/null
!pip install -q -U . yt-dlp 2>&1 | grep -v -E 'WARNING|already satisfied' || true
if HAS_GPU:
    # CUDA 12 runtime libs for the speech model (Colab's system CUDA may be a different version)
    !pip install -q nvidia-cublas-cu12 'nvidia-cudnn-cu12>=9,<10' 2>&1 | grep -v -E 'WARNING|already satisfied' || true

if HAS_GPU:
    print('✅ Installed. GPU ready:', gpu.split(' (UUID')[0])
else:
    print('✅ Installed.')
    print('⚠️  NO GPU — everything will be MUCH slower (a 1-hour video can take an hour or more).')
    print('   Fix: Runtime ▸ Change runtime type ▸ T4 GPU ▸ Save, then Runtime ▸ Run all again.')
    print('   (If Colab says no GPU is available, Google has hit its free limit — try again later,')
    print('    or keep going on CPU with a short video or a small minutes_to_use.)')


In [ ]:
#@title 2️⃣ Settings
#@markdown Paste a YouTube link with **people talking** (podcast, interview, vlog, stream replay, lecture). Only clip videos you have the rights to.
youtube_url = ""  #@param {type:"string"}
#@markdown Or tick this to upload a video file from your computer instead (use it if YouTube blocks the download):
upload_my_own_video = False  #@param {type:"boolean"}
caption_style = "hormozi"  #@param ["hormozi", "mrbeast", "neon", "boxed", "minimal"]
number_of_clips = 3  #@param {type:"slider", min:1, max:10, step:1}
framing = "crop"  #@param ["crop", "blur", "fit"]
#@markdown **Long streams:** only the first N minutes are used, so it finishes in minutes instead of hours. `0` = whole video.
minutes_to_use = 60  #@param {type:"integer"}
whisper_model = "small"  #@param ["tiny", "base", "small", "medium", "large-v3"]
#@markdown **Optional – smarter highlight picking & captions.** Leave blank to use the built-in scorer (no key needed).
anthropic_api_key = ""  #@param {type:"string"}
openai_api_key = ""  #@param {type:"string"}

import os
if anthropic_api_key: os.environ['ANTHROPIC_API_KEY'] = anthropic_api_key.strip()
if openai_api_key: os.environ['OPENAI_API_KEY'] = openai_api_key.strip()
LLM = 'auto' if (anthropic_api_key or openai_api_key) else 'none'
print('✅ Settings saved.')
print('   Video:', 'upload from your computer' if upload_my_own_video else (youtube_url.strip() or '⚠️ no link yet — paste one above'))
print('   Length used:', f'first {minutes_to_use} min' if minutes_to_use else 'whole video')
print('   Highlight engine:', 'AI (your API key)' if LLM == 'auto' else 'built-in scorer (free)')


In [ ]:
#@title 3️⃣ Make the clips 🎬
import os, time, shutil, subprocess
from clipforge.config import settings
from clipforge.pipeline import new_job, run_job

# Apply the settings from step 2 (works even if you changed them after a first run)
settings.work_dir = __import__('pathlib').Path('/content/clipforge_output')
settings.llm_provider = LLM
settings.whisper_model = whisper_model
try:
    _gpu = bool(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True, timeout=20).stdout.strip())
except (FileNotFoundError, subprocess.SubprocessError, OSError):
    _gpu = False  # CPU-only runtime
if not _gpu and whisper_model in ('small', 'medium', 'large-v3'):
    settings.whisper_model = 'base'
    print('ℹ️  No GPU, so using the faster "base" speech model instead of', repr(whisper_model))

source = youtube_url.strip()
if upload_my_own_video:
    from google.colab import files
    print('📁 Pick a video file from your computer (big files take a while to upload)…')
    up = files.upload()
    if not up:
        raise SystemExit('No file was uploaded.')
    name = next(iter(up))
    os.makedirs('/content/uploads', exist_ok=True)
    source = os.path.join('/content/uploads', name)
    shutil.move(name, source)
if not source:
    raise SystemExit('👆 Paste a YouTube link into youtube_url in step 2️⃣ (or tick upload_my_own_video), run step 2️⃣, then run this one again.')

print('⏳ Working — progress updates every few seconds. Long videos take longer; keep this tab open.\n')
job = new_job(source, style=caption_style, clip_count=number_of_clips, reframe=framing,
              max_minutes=minutes_to_use or None)
_t0 = time.time()
_last = {'s': None, 't': 0.0, 'm': None}
def _show(j):
    now = time.time()
    if j.status != _last['s'] or (j.message != _last['m'] and now - _last['t'] >= 4):
        el = int(now - _t0)
        print(f"[{el // 60:02d}:{el % 60:02d}] {j.status:<12} {j.progress * 100:5.1f}%  {j.message}", flush=True)
        _last.update(s=j.status, t=now, m=j.message)
run_job(job, _show)

if job.status == 'error':
    raise SystemExit(f'\n❌ {job.error}')
print(f'\n✅ {len(job.clips)} clips ready in {int(time.time() - _t0)} s — job {job.id}')
for c in job.clips:
    h = c.highlight
    print(f"  {c.index}. [{h.start:6.1f}s → {h.end:6.1f}s]  🔥{h.score:3.0f}  {h.title}")


In [ ]:
#@title 4️⃣ Preview clips + captions
from IPython.display import HTML, display
import base64, html

for c in job.clips:
    data = base64.b64encode(open(c.video_path, 'rb').read()).decode()
    post = c.posts.get('tiktok')
    cap = html.escape(post.full_caption if post else '')
    display(HTML(f'''
    <div style="display:flex;flex-wrap:wrap;gap:16px;align-items:flex-start;margin:12px 0;font-family:sans-serif">
      <video controls width="240" style="border-radius:12px;background:#000" src="data:video/mp4;base64,{data}"></video>
      <div style="flex:1;min-width:260px"><h3 style="margin:0 0 6px">{c.index}. {html.escape(c.highlight.title)}</h3>
      <p style="color:#888;margin:0 0 8px">{c.highlight.start:.0f}s → {c.highlight.end:.0f}s · {c.highlight.mood}</p>
      <pre style="white-space:pre-wrap;background:#1a1a1a;color:#eee;padding:10px;border-radius:8px;max-width:520px">{cap}</pre></div>
    </div>'''))

In [ ]:
#@title 5️⃣ Download everything (MP4s + SRTs + captions for every platform)
import json, shutil, os
from google.colab import files

out = f'/content/clipforge_output/clips/{job.id}'
with open(os.path.join(out, 'captions.md'), 'w') as f:
    for c in job.clips:
        f.write(f'# Clip {c.index}: {c.highlight.title}\n\n')
        for p, post in c.posts.items():
            f.write(f'## {p}\n**{post.title}**\n\n{post.full_caption}\n\n')
zip_path = shutil.make_archive(f'/content/clipforge_{job.id}', 'zip', out)
print('📦', zip_path)
files.download(zip_path)

## Posting

Colab has no desktop browser, so the one-click **Open in TikTok →** publisher isn't available here — download the zip, then upload the MP4s in each app and paste the matching caption from `captions.md`.

For the full experience (web dashboard + browser publishing to TikTok / Shorts / Reels / Facebook / X), install locally:

```bash
git clone https://github.com/WHERESHOULDISTART/clipforge && cd clipforge
pip install -e . && playwright install chromium
clipforge serve
```

⭐ Star the repo if this saved you time: https://github.com/WHERESHOULDISTART/clipforge